In [ ]:
# necessary libraries
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from datetime import datetime

# Loading the tables!!

In [ ]:
df_master = pd.read_csv('/Users/kishohars/Projects/football_valuation_project/data_clean/df_master.csv')
df_appearances = pd.read_csv('data/raw/dataset/appearances.csv')

In [ ]:
df_appearances.info()

In [ ]:
df_appearances.head()

In [ ]:
# Group by player_id and aggregate key stats!!

df_perf = df_appearances.groupby('player_id', as_index=False).agg({
    'minutes_played': 'sum',
    'goals': 'sum',
    'assists': 'sum',
    'yellow_cards': 'sum',
    'red_cards': 'sum'
})

# Calculate per-90 stats safely
df_perf = df_perf[df_perf['minutes_played'] > 0]  # remove players with 0 minutes
df_perf['goals_per90'] = df_perf['goals'] / (df_perf['minutes_played'] / 90)
df_perf['assists_per90'] = df_perf['assists'] / (df_perf['minutes_played'] / 90)

# Replace NaN or inf values with 0
df_perf.replace([float('inf'), -float('inf')], 0, inplace=True)
df_perf.fillna(0, inplace=True)

In [ ]:
df_perf.head()

In [ ]:
df_perf.columns.tolist()

# Combining our 'df_master' table and newly accuried 'df_perf' table with player_id as our primary key!!

In [ ]:
df_whole = pd.merge(df_master, df_perf, how = 'left', left_on = 'player_id', right_on = 'player_id')


df_whole.info()

In [ ]:
df_whole.head()

In [ ]:
# Finding who got the most red cards in our dataset!!

df_whole.pivot_table(index = 'name', values = 'red_cards', aggfunc = 'sum').sort_values(by = 'red_cards', ascending = False).head(20).plot(kind = 'bar', figsize = (20, 10))
sns.set_style('dark')
plt.xticks(rotation = 45)
plt.title('Top 20 players with most red cards')
plt.ylabel('Number of Red Cards')
plt.xlabel('Player Name')
plt.tight_layout()
plt.show()

In [ ]:
# Finding who got the most goals in our dataset!!

df_whole.pivot_table(index = 'name', values = 'goals', aggfunc = 'sum').sort_values(by = 'goals', ascending = False).head(20).plot(kind = 'bar', figsize = (20, 10))
plt.xticks(rotation = 45)
plt.title('Top 20 Goal Scorers in our Dataset')
plt.xlabel('Player Name')
plt.ylabel('Goals Scored')
plt.tight_layout()
plt.show()

In [ ]:
# Finding who made the most assists in our dataaset!!

df_whole.pivot_table(index = 'name', values = 'assists', aggfunc = 'sum').sort_values(by ='assists', ascending = False).head(20).plot(kind = 'bar', figsize = (20, 10))
plt.xlabel('Player Name')
plt.ylabel('Assists Made')
plt.title('Top 20 Players with most Assists')
plt.xticks(rotation = 45)
plt.tight_layout()

In [ ]:
df_whole.to_csv('/Users/kishohars/Projects/football_valuation_project/data_clean/df_whole.csv', index = False)